# Table 14 — Actual-image inference latency, throughput, and GPU memory (NO CHECKPOINTS)

**Source:** original Kaggle garbage-classification dataset used in the Stage 2 REMNET notebooks, with the audited **463-image Testing cohort**. The same program also accepts the original Google Drive ImageFolder path from `Latency.ipynb` if running in Colab, but the locked 463-image cohort check is intended for the original Kaggle dataset.

**Models:** MobileNetV2, EfficientNet-Lite0, ResNet50, and equal-weight softmax ensemble REMNET. **Input:** RGB, **32×32**, normalization mean/std 0.5. **Batch-1 latency:** host-observed synchronized model-forward time; **batch-32 throughput:** forward-only; **memory:** peak PyTorch GPU allocation including model parameters and activations. Five repeated profiling trials (not trained model seeds).

> **Critical interpretation:** This benchmark loads **no trained checkpoints** and does **not train**. All model parameters are freshly initialized (`weights=None`, `pretrained=False`), therefore the outputs describe **architecture-level inference performance only**. Accuracy, precision, recall, and F1 are **not valid** and are intentionally not computed. Do not present these numbers as measured performance of the already-trained Stage 2 model, because different trained weights or implementation settings can affect speed. For production-trained-model latency, trained weights would need to be used.

> **Real-time status** means *inference-only* batch-1 **p95 latency ≤ 33.33 ms (30 fps)** on the measured hardware. It does not establish end-to-end camera, image loading, preprocessing, or device-deployment real-time capability.

## 1. Dependencies, source paths, and benchmark settings

The first path is taken from your `r12.zip` Stage 2 notebooks; the Colab paths come from your uploaded `Latency.ipynb`. Attach the original dataset in Kaggle or mount Google Drive in Colab. No checkpoint path is used.

In [3]:
# Install timm if necessary. On Kaggle, make sure timm is available.
import sys, subprocess, importlib.util
if importlib.util.find_spec('timm') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])

import os, json, csv, gc, time, math, random, hashlib, zipfile, platform
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models, transforms
from PIL import Image
import timm
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

# Original path from r12.zip Stage 2 notebook (primary).
DATASET_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset'),
    Path('/kaggle/input/garbage-classification-image-dataset'),
    # Original path in uploaded Latency.ipynb (when running in Google Colab):
    Path('/content/drive/MyDrive/dataset'),
]
# In Colab only, mount Drive in a separate cell BEFORE this one:
# from google.colab import drive
# drive.mount('/content/drive')

OUTPUT_DIR = Path('/kaggle/working/REMNET_Table14_No_Checkpoints') if Path('/kaggle/working').is_dir() else Path('./REMNET_Table14_No_Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_RESOLUTION = 32    # Original Stage 2 executable transforms
BATCH_THROUGHPUT = 32
BATCH_LATENCY = 1
PROFILING_REPEATS = 5   # Five timing repeats, NOT five training seeds
WARMUP_BATCHES = 8
BATCH1_LIMIT = 128       # First 128 original TEST images, per repeat; set None for all 463
THROUGHPUT_LIMIT = None  # Entire audited original TEST cohort (463) per repeat
NUM_WORKERS = 0          # Reliable across Kaggle and Google Drive
REQUIRE_CUDA = True      # Prevent mixing GPU and CPU performance in a single Table 14
REALTIME_THRESHOLD_MS = 1000.0 / 30.0
STRICT_AUDITED_463 = True  # Refuse altered datasets / different test membership
INIT_SEED = 7            # Fixed deterministic initialization; not trained

random.seed(INIT_SEED)
np.random.seed(INIT_SEED)
torch.manual_seed(INIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(INIT_SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# Disable TF32 for a simple consistent FP32 benchmark.
if hasattr(torch.backends, 'cuda'):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.allow_tf32 = False

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if REQUIRE_CUDA and device.type != 'cuda':
    raise RuntimeError('Enable Kaggle GPU / Colab GPU before running the benchmark.')

print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'timm:', timm.__version__)
print('Output directory:', OUTPUT_DIR)


Device: cuda:0
GPU: Tesla T4
torch: 2.11.0+cu130 torchvision: 0.26.0+cu130 timm: 1.0.30
Output directory: /kaggle/working/REMNET_Table14_No_Checkpoints


## 2. Find original Training/Testing folders and reproduce audited Stage 2 test membership

The source folder can be an extracted directory or ZIP archive. Only the original `Testing` images are **evaluated**. Training images are read for a checksum-based de-duplication audit, not for training or model selection. The original Stage 2 notebook identifies **463** testing images after exact-hash de-duplication and exclusion of conflicting-label groups.

In [4]:
IMG_EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}

root = next((p for p in DATASET_ROOT_CANDIDATES if p.is_dir()), None)
if root is None:
    raise FileNotFoundError('Original dataset not found. Attach the Kaggle dataset (or mount original Colab Drive dataset). Tried: '+str(DATASET_ROOT_CANDIDATES))
print('Original dataset root:',root)

def guarded_unzip(archive:Path,target:Path):
    marker=target/'.unpacked.complete'
    if marker.is_file(): return
    target.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            dest=(target/entry.filename).resolve()
            if not dest.is_relative_to(target.resolve()):
                raise RuntimeError(f'Unsafe ZIP entry: {entry.filename}')
        z.extractall(target)
    marker.touch()

roots=[root]
archives=sorted(p for p in root.rglob('*.zip') if p.is_file())
for ix,p in enumerate(archives):
    dest=OUTPUT_DIR/'unpacked_original_source'/f'{ix:02d}_{p.stem}'
    print('Extracting source archive:',p)
    guarded_unzip(p,dest)
    roots.append(dest)

found=[]
for r in roots:
    for parent in [r,*sorted((p for p in r.rglob('*') if p.is_dir()),key=str)]:
        ch={p.name.lower():p for p in parent.iterdir() if p.is_dir()}
        tr=ch.get('training') or ch.get('train')
        te=ch.get('testing') or ch.get('test')
        if tr and te:
            tr_classes={p.name for p in tr.iterdir() if p.is_dir()}
            te_classes={p.name for p in te.iterdir() if p.is_dir()}
            if tr_classes and tr_classes==te_classes:
                found.append((tr,te))
if not found:
    raise FileNotFoundError('No matching Training/Testing ImageFolder directories were found under '+str(root))
TRAIN_DIR,TEST_DIR=found[0]
CLASS_NAMES=sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
if len(CLASS_NAMES)!=6:
    raise ValueError(f'Expected six source classes; found {CLASS_NAMES}')
print('Training:',TRAIN_DIR)
print('Testing:',TEST_DIR)
print('Classes:',CLASS_NAMES)

def sha256_file(path:Path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

hash_groups=defaultdict(list)
for partition,base in [('Training',TRAIN_DIR),('Testing',TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((base/cls).iterdir()):
            if path.is_file() and path.suffix.lower() in IMG_EXTS:
                hash_groups[sha256_file(path)].append({
                  'path':path, 'class':cls,'source':f'{partition}/{cls}/{path.name}',
                  'original_partition':partition})

cleaned=[]; conflicting_file_count=0
for digest,items in hash_groups.items():
    if len({x['class'] for x in items})!=1:
        conflicting_file_count+=len(items)
        continue
    item=min(items,key=lambda x:x['source']) # same rule as r12 Stage 2
    cleaned.append({**item,'sha256':digest})

original_test=sorted((r for r in cleaned if r['original_partition']=='Testing'),key=lambda x:x['source'])
original_train=sorted((r for r in cleaned if r['original_partition']=='Training'),key=lambda x:x['sha256'])
print('Audited source: cleaned_total=',len(cleaned),'; train_pool=',len(original_train),'; test=',len(original_test),'; conflicting files=',conflicting_file_count)
if STRICT_AUDITED_463 and (len(cleaned),len(original_train),len(original_test),conflicting_file_count)!=(2910,2447,463,6):
    raise RuntimeError('Source differs from the original r12 Stage 2 audited dataset. Do not silently report this as the same 463-image cohort.')
if not original_test:
    raise RuntimeError('No test images found.')

manifest=pd.DataFrame([
    {'source_path':r['source'],'class':r['class'],'sha256':r['sha256']}
    for r in original_test
]).sort_values('source_path')
manifest_path=OUTPUT_DIR/'original_test_audited_manifest.csv'
manifest.to_csv(manifest_path,index=False)
test_fingerprint=hashlib.sha256(manifest.to_csv(index=False).encode('utf-8')).hexdigest()
print('Test images:',len(original_test),'| test manifest SHA-256:',test_fingerprint)
print('Per-class test count:')
print(manifest['class'].value_counts().sort_index().to_string())


Original dataset root: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset
Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training
Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Audited source: cleaned_total= 2910 ; train_pool= 2447 ; test= 463 ; conflicting files= 6
Test images: 463 | test manifest SHA-256: af7556bafaa692a27f2aa56e4bb0bd2b2e49e68e7625ff0e994c5bc349442826
Per-class test count:
class
cardboard    67
glass        84
metal        84
paper        96
plastic      83
trash        49


## 3. Load real test images (no training)

The *same* original test images and deterministic preprocessing are used for every architecture. Labels are available solely for manifest integrity; this benchmark deliberately does not calculate classification accuracy for randomly initialized weights.

In [5]:
transform=transforms.Compose([
    transforms.Resize((IMAGE_RESOLUTION,IMAGE_RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])
class OriginalTestDataset(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row=self.rows[index]
        with Image.open(row['path']) as im:
            x=transform(im.convert('RGB'))
        return x, CLASS_NAMES.index(row['class'])

dataset=OriginalTestDataset(original_test)

def make_loader(batch_size):
    return DataLoader(dataset,batch_size=batch_size,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=(device.type=='cuda'),drop_last=False)

print('Real test images loaded:',len(dataset))
print('Benchmark image preprocessing:',IMAGE_RESOLUTION,'×',IMAGE_RESOLUTION,'RGB / FP32')


Real test images loaded: 463
Benchmark image preprocessing: 32 × 32 RGB / FP32


## 4. Build the four model architectures — no checkpoints, no downloads, no training

REMNET uses the original Stage 2 fusion rule: an **equal-weight mean of three softmax probability vectors**. The benchmarking models are initialized afresh to avoid importing any saved trained weights.

In [6]:
NUM_CLASSES=len(CLASS_NAMES)

def create_model(name):
    if name=='MobileNetV2':
        m=models.mobilenet_v2(weights=None)
        m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
    elif name=='EfficientNet-Lite0':
        m=timm.create_model('efficientnet_lite0',pretrained=False,num_classes=NUM_CLASSES)
    elif name=='ResNet50':
        m=models.resnet50(weights=None)
        m.fc=nn.Linear(m.fc.in_features,NUM_CLASSES)
    elif name=='Ensemble REMNET':
        return REMNETEnsemble(
            create_model('MobileNetV2'),
            create_model('EfficientNet-Lite0'),
            create_model('ResNet50')
        )
    else:
        raise ValueError(name)
    return m

class REMNETEnsemble(nn.Module):
    def __init__(self,mobile,lite,resnet):
        super().__init__()
        self.mobile=mobile
        self.lite=lite
        self.resnet=resnet
    def forward(self,x):
        return (F.softmax(self.mobile(x),dim=1) +
                F.softmax(self.lite(x),dim=1) +
                F.softmax(self.resnet(x),dim=1)) / 3.0

MODEL_NAMES=['MobileNetV2','EfficientNet-Lite0','ResNet50','Ensemble REMNET']
print('Models:',MODEL_NAMES)


Models: ['MobileNetV2', 'EfficientNet-Lite0', 'ResNet50', 'Ensemble REMNET']


## 5. Synchronized batch-1 latency and batch-32 throughput with GPU memory

Model forward computation is timed; image loading, CPU preprocessing, and host-to-device transfers are **excluded**. For each repeat, a batch is transferred to GPU first, CUDA synchronized, the model forward is timed using `time.perf_counter()`, and CUDA is synchronized again. This approach captures CPU launch overhead and synchronized GPU execution. After warm-up, PyTorch's peak memory tracker is reset for each test. Peak memory is the **allocated GPU memory of the model and activations**, not a whole-device VRAM or system-RSS measurement.

The repeated trials are **profiling repeats**, not training seeds. Mean and sample SD refer to profiling-repeat variability. Batch-1 `p95` is obtained from the distribution of individual image times pooled over repeats.

In [7]:
def cuda_sync():
    if device.type=='cuda':
        torch.cuda.synchronize()

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def batch_iterator(loader,limit=None):
    count=0
    for images,_ in loader:
        if limit is not None:
            rem=limit-count
            if rem<=0: break
            images=images[:rem]
        if len(images)==0: break
        yield images
        count+=len(images)

def warm_up(model,loader):
    batches=[]
    for images in batch_iterator(loader):
        batches.append(images.to(device,non_blocking=True))
        if len(batches)>=min(WARMUP_BATCHES,4): break
    if not batches:
        raise RuntimeError('Empty test loader')
    with torch.inference_mode():
        for j in range(WARMUP_BATCHES):
            x=batches[j%len(batches)]
            _=model(x)
    cuda_sync()

def measure_forward(model,loader,limit,metric_name):
    warm_up(model,loader)
    if device.type=='cuda':
        torch.cuda.reset_peak_memory_stats(device)
    dt=[];count=0
    with torch.inference_mode():
        for images in batch_iterator(loader,limit):
            x=images.to(device,non_blocking=True)
            cuda_sync()
            t0=time.perf_counter()
            output=model(x)
            cuda_sync()
            elapsed=time.perf_counter()-t0
            if (not torch.isfinite(output).all()):
                raise RuntimeError(f'{metric_name}: non-finite output')
            dt.append((elapsed,len(x)))
            count+=len(x)
            del output,x
    if not dt:
        raise RuntimeError(f'No data for {metric_name}')
    total_seconds=sum(sec for sec,n in dt)
    peak_allocated_mb=(torch.cuda.max_memory_allocated(device)/1e6) if device.type=='cuda' else float('nan')
    return {'seconds':total_seconds,'images':count,'peak_allocated_mb':peak_allocated_mb,
            'ms_per_image':(1000*total_seconds/count),
            'images_per_second':(count/total_seconds),
            'batch_times_ms':[sec*1000 for sec,n in dt]}

all_rows=[]
all_batch1_latencies=[]
for model_name in MODEL_NAMES:
    print('\n'+'='*72)
    print('Benchmark:',model_name)
    # Seed is fixed; repeat measurement on the SAME initialized architecture.
    torch.manual_seed(INIT_SEED)
    if device.type=='cuda': torch.cuda.manual_seed_all(INIT_SEED)
    model=create_model(model_name).to(device).eval()
    params=count_params(model)
    loader1=make_loader(BATCH_LATENCY)
    loader32=make_loader(BATCH_THROUGHPUT)
    for repeat in range(1,PROFILING_REPEATS+1):
        # Batch=1 gives one forward time per image, suitable for p95 latency.
        a=measure_forward(model,loader1,BATCH1_LIMIT,'single-image latency')
        b=measure_forward(model,loader32,THROUGHPUT_LIMIT,'batch throughput')
        for latency in a['batch_times_ms']:
            all_batch1_latencies.append({'model':model_name,'profiling_repeat':repeat,'latency_ms':latency})
        row={
            'model':model_name,'profiling_repeat':repeat,'input_resolution':IMAGE_RESOLUTION,
            'batch1_num_images':a['images'],'batch32_num_images':b['images'],
            'batch1_latency_ms_mean':a['ms_per_image'],
            'batch1_latency_ms_p95':float(np.percentile(a['batch_times_ms'],95)),
            'batch32_throughput_img_s':b['images_per_second'],
            'batch32_amortized_ms_img':b['ms_per_image'],
            'peak_memory_batch1_mb':a['peak_allocated_mb'],
            'peak_memory_batch32_mb':b['peak_allocated_mb'],
            'parameters_millions':params/1e6,
            'gpu':torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU',
            'weights':'random initialization; no checkpoints; no training',
        }
        all_rows.append(row)
        print('Repeat',repeat,
              f"B1 mean={row['batch1_latency_ms_mean']:.3f} ms",
              f"B1 p95={row['batch1_latency_ms_p95']:.3f} ms",
              f"B32 throughput={row['batch32_throughput_img_s']:.2f} img/s",
              f"B32 peak={row['peak_memory_batch32_mb']:.1f} MB")
    del model,loader1,loader32
    gc.collect()
    if device.type=='cuda':
        torch.cuda.empty_cache()
        cuda_sync()

runs=pd.DataFrame(all_rows)
runs.to_csv(OUTPUT_DIR/'Table14_per_repeat_measurements.csv',index=False,float_format='%.8f')
raw_latencies=pd.DataFrame(all_batch1_latencies)
raw_latencies.to_csv(OUTPUT_DIR/'Table14_batch1_latency_samples.csv',index=False,float_format='%.8f')



Benchmark: MobileNetV2
Repeat 1 B1 mean=5.458 ms B1 p95=6.194 ms B32 throughput=3043.59 img/s B32 peak=25.9 MB
Repeat 2 B1 mean=5.656 ms B1 p95=6.456 ms B32 throughput=4561.98 img/s B32 peak=25.9 MB
Repeat 3 B1 mean=5.603 ms B1 p95=6.533 ms B32 throughput=5302.24 img/s B32 peak=25.9 MB
Repeat 4 B1 mean=5.388 ms B1 p95=6.236 ms B32 throughput=4959.50 img/s B32 peak=25.9 MB
Repeat 5 B1 mean=5.498 ms B1 p95=6.392 ms B32 throughput=5074.43 img/s B32 peak=25.9 MB

Benchmark: EfficientNet-Lite0
Repeat 1 B1 mean=5.571 ms B1 p95=6.372 ms B32 throughput=4241.65 img/s B32 peak=31.6 MB
Repeat 2 B1 mean=5.555 ms B1 p95=6.318 ms B32 throughput=5011.45 img/s B32 peak=31.6 MB
Repeat 3 B1 mean=5.507 ms B1 p95=6.297 ms B32 throughput=4659.12 img/s B32 peak=31.6 MB
Repeat 4 B1 mean=5.542 ms B1 p95=6.250 ms B32 throughput=4785.60 img/s B32 peak=31.6 MB
Repeat 5 B1 mean=5.579 ms B1 p95=6.353 ms B32 throughput=4801.57 img/s B32 peak=31.6 MB

Benchmark: ResNet50
Repeat 1 B1 mean=6.771 ms B1 p95=7.981 ms B3

## 6. Create Table 14, five-repeat variability summary and reproducibility metadata

**Memory (MB)** is total peak PyTorch allocated memory during batch-32 inference. **Batch-1 latency** is not the reciprocal of batch-32 throughput. A real-time designation is based on **batch-1 95th-percentile latency**, and is limited to GPU forward inference.

In [8]:
table_rows=[];summary_rows=[]
for name in MODEL_NAMES:
    sub=runs.loc[runs['model']==name].copy()
    all_lat=raw_latencies.loc[raw_latencies['model']==name,'latency_ms'].to_numpy()
    latency_mean=float(np.mean(all_lat))
    latency_p95=float(np.percentile(all_lat,95))
    throughput_mean=float(sub['batch32_throughput_img_s'].mean())
    throughput_sd=float(sub['batch32_throughput_img_s'].std(ddof=1))
    mem_mean=float(sub['peak_memory_batch32_mb'].mean())
    mem_sd=float(sub['peak_memory_batch32_mb'].std(ddof=1))
    realtime=('Real-Time Capable (inference-only)'
              if latency_p95<=REALTIME_THRESHOLD_MS else 'Not Real-Time (inference-only)')
    table_rows.append({
        'Model':name,
        'Latency (ms/img; B1 mean)':round(latency_mean,4),
        'Latency p95 (ms; B1)':round(latency_p95,4),
        'Throughput (img/sec; B32)':round(throughput_mean,4),
        'Peak GPU memory (MB; B32)':round(mem_mean,4),
        'Real-Time Status (30 FPS, B1 p95)':realtime,
    })
    summary_rows.append({
        'model':name,'profiling_repeats':len(sub),'single_image_samples':len(all_lat),
        'batch1_mean_ms':latency_mean,
        'batch1_p95_ms':latency_p95,
        'repeat_mean_batch1_latency_ms_sd':float(sub['batch1_latency_ms_mean'].std(ddof=1)),
        'batch32_throughput_img_s_mean':throughput_mean,
        'batch32_throughput_img_s_sd':throughput_sd,
        'batch32_peak_gpu_memory_mb_mean':mem_mean,
        'batch32_peak_gpu_memory_mb_sd':mem_sd,
        'parameters_millions':float(sub['parameters_millions'].iloc[0]),
        'real_time_status':realtime})

table14=pd.DataFrame(table_rows)
summary=pd.DataFrame(summary_rows)
print('FINAL TABLE 14 (no checkpoints; architecture-level GPU performance):')
display(table14)
print('FIVE-PROFILING-REPEAT VARIABILITY:')
display(summary)

table14.to_csv(OUTPUT_DIR/'Table14_Original_Dataset_No_Checkpoints.csv',index=False)
summary.to_csv(OUTPUT_DIR/'Table14_FiveRepeat_Summary.csv',index=False,float_format='%.8f')
metadata={
 'task':'Inference-only architecture benchmarking, not trained-model performance',
 'original_dataset_root':str(root),'test_root':str(TEST_DIR),
 'test_cohort_size':len(original_test),'test_manifest_sha256':test_fingerprint,
 'classes':CLASS_NAMES,'preprocessing':f'RGB Resize({IMAGE_RESOLUTION},{IMAGE_RESOLUTION}) Normalize(mean=.5,std=.5)',
 'models':MODEL_NAMES,'weights_loaded':False,'trained':False,'input_resolution':IMAGE_RESOLUTION,
 'profiling_repeats':PROFILING_REPEATS,'batch1_images_per_repeat':runs['batch1_num_images'].unique().tolist(),
 'batch32_images_per_repeat':runs['batch32_num_images'].unique().tolist(),
 'batch1_definition':'host-clock forward only, synchronized CUDA, p95 of individual forward times',
 'throughput_definition':'total input images / synchronized batch-32 model-forward seconds; excludes decoding and transfer',
 'memory_definition':'max PyTorch GPU allocated memory, includes weights and activations, 1 MB = 1e6 bytes',
 'real_time_definition':f'B1 p95 <= {REALTIME_THRESHOLD_MS:.4f} ms; inference-only',
 'device':str(device),'gpu_name':torch.cuda.get_device_name(0) if device.type=='cuda' else None,
 'torch_version':torch.__version__,'torchvision_version':torchvision.__version__,'timm_version':timm.__version__,
 'cuda_version':torch.version.cuda,'system':platform.platform(),
 'warning':'No checkpoints, no training. No valid accuracy metrics. Profiling repeats are not trained model seeds.'
}
(OUTPUT_DIR/'benchmark_metadata.json').write_text(json.dumps(metadata,indent=2,default=str),encoding='utf-8')

zip_path=OUTPUT_DIR.parent/'REMNET_Table14_Original_Dataset_No_Checkpoints_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.glob('*')):
        if p.is_file(): z.write(p,arcname=p.name)
print('Files saved at:',OUTPUT_DIR)
print('ZIP:',zip_path)
print('Important: these measurements describe UNTRAINED architectural inference, not the trained D1 checkpoints.')


FINAL TABLE 14 (no checkpoints; architecture-level GPU performance):


,Model,Latency (ms/img; B1 mean),Latency p95 (ms; B1),Throughput (img/sec; B32),Peak GPU memory (MB; B32),"Real-Time Status (30 FPS, B1 p95)"
0,MobileNetV2,5.5205,6.4302,4588.3485,25.9215,Real-Time Capable (inference-only)
1,EfficientNet-Lite0,5.5506,6.3305,4699.8765,31.5781,Real-Time Capable (inference-only)
2,ResNet50,6.6527,7.9160,2810.2080,149.1169,Real-Time Capable (inference-only)
3,Ensemble REMNET,18.4097,20.0717,1475.7058,170.5728,Real-Time Capable (inference-only)


FIVE-PROFILING-REPEAT VARIABILITY:


,model,profiling_repeats,single_image_samples,batch1_mean_ms,batch1_p95_ms,repeat_mean_batch1_latency_ms_sd,batch32_throughput_img_s_mean,batch32_throughput_img_s_sd,batch32_peak_gpu_memory_mb_mean,batch32_peak_gpu_memory_mb_sd,parameters_millions,real_time_status
0,MobileNetV2,5,640,5.520457,6.430249,0.108298,4588.348507,904.246484,25.921536,3.972055e-15,2.231558,Real-Time Capable (inference-only)
1,EfficientNet-Lite0,5,640,5.550643,6.330469,0.028434,4699.876511,285.657593,31.578112,3.972055e-15,3.378694,Real-Time Capable (inference-only)
2,ResNet50,5,640,6.652702,7.916006,0.114539,2810.208037,172.201005,149.116928,0.000000e+00,23.520326,Real-Time Capable (inference-only)
3,Ensemble REMNET,5,640,18.409675,20.071724,0.453200,1475.705831,47.628707,170.572800,0.000000e+00,29.130578,Real-Time Capable (inference-only)


Files saved at: /kaggle/working/REMNET_Table14_No_Checkpoints
ZIP: /kaggle/working/REMNET_Table14_Original_Dataset_No_Checkpoints_Outputs.zip
Important: these measurements describe UNTRAINED architectural inference, not the trained D1 checkpoints.


### Reporting guidance

The CSV titled **`Table14_Original_Dataset_No_Checkpoints.csv`** can be copied into the manuscript as an *architecture-only* latency/memory experiment, provided the source dataset and hardware details are disclosed. It **must not** be substituted for measured deployment behavior of the trained REMNET checkpoints without an appropriate caveat. This notebook uses neither checkpoint loading nor retraining. No accuracy or CO₂/energy metrics are generated; those require separate, appropriately controlled experiments.

If you want Table 14 to represent the **actual trained models** rather than their architectures, the benchmark necessarily has to load their trained weights or train new models first.